# 01 — Data Exploration
**Week 2 deliverable: California Property Close Price Prediction**

Goal: load at least 6 months of `CRMLSSold` files, restrict to
`PropertyType = "Residential"` and `PropertySubType = "SingleFamilyResidence"`,
and explore the target (`ClosePrice`) and key features.

> Before running: check `Trestle Property MetaData.pdf` and rename any columns below
> that don't match your actual files (column names can vary slightly by MLS export).


In [ ]:
import glob
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


## 1. Load all downloaded CRMLSSold files (6+ months)

In [ ]:
# Point this at the local folder where you downloaded the CRMLSSold files via FileZilla
DATA_DIR = "./data/raw"  # <-- update to your local path

files = sorted(glob.glob(os.path.join(DATA_DIR, "CRMLSSold*")))
print(f"Found {len(files)} files:")
for f in files:
    print(" -", os.path.basename(f))

assert len(files) >= 6, "Task requires a minimum of 6 months of data — check DATA_DIR."


In [ ]:
dfs = []
for f in files:
    tmp = pd.read_csv(f, low_memory=False)
    tmp["__source_file"] = os.path.basename(f)
    dfs.append(tmp)

df_raw = pd.concat(dfs, ignore_index=True)
print(df_raw.shape)
df_raw.head()


## 2. Filter to Residential / Single Family per task spec

In [ ]:
df = df_raw[
    (df_raw["PropertyType"] == "Residential")
    & (df_raw["PropertySubType"] == "SingleFamilyResidence")
].copy()

print(f"Rows before filter: {len(df_raw):,}")
print(f"Rows after filter:  {len(df):,}")


## 3. Structure check
Shape, dtypes, missing values — this drives your Week 3 preprocessing decisions.

In [ ]:
df.info()


In [ ]:
missing = (
    df.isna().mean().sort_values(ascending=False) * 100
).round(2)
missing[missing > 0].to_frame("pct_missing")


## 4. Target variable: ClosePrice

In [ ]:
print(df["ClosePrice"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["ClosePrice"], bins=60, ax=axes[0])
axes[0].set_title("ClosePrice distribution")

sns.histplot(np.log1p(df["ClosePrice"]), bins=60, ax=axes[1])
axes[1].set_title("log1p(ClosePrice) distribution")
plt.tight_layout()
plt.show()


ClosePrice is almost always right-skewed with a long tail of high-end sales.
Note in your writeup whether a log transform looks more model-friendly than raw price.

## 5. Key features: LivingArea, Bedrooms, Bathrooms, LotSize

In [ ]:
# Adjust these names if your metadata doc uses different labels
feature_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, feature_cols):
    sns.histplot(df[col].dropna(), bins=50, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
df[feature_cols + ["ClosePrice"]].describe()


## 6. Outlier check (boxplots)

In [ ]:
fig, axes = plt.subplots(1, len(feature_cols) + 1, figsize=(16, 4))
for ax, col in zip(axes, feature_cols + ["ClosePrice"]):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


Flag anything that looks like a data-entry error (e.g. 0-bedroom "single family"
homes, LotSize of 0, ClosePrice near $0 or in the tens of millions) — decide how to
handle these in Week 3, don't drop them yet without a reason.

## 7. Relationship of each feature to ClosePrice

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, feature_cols):
    sns.scatterplot(x=df[col], y=df["ClosePrice"], ax=ax, alpha=0.3, s=10)
    ax.set_title(f"ClosePrice vs {col}")
plt.tight_layout()
plt.show()


## 8. Correlation heatmap (numeric features)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr[["ClosePrice"]].sort_values("ClosePrice", ascending=False),
            annot=True, cmap="coolwarm", center=0)
plt.title("Correlation with ClosePrice")
plt.show()


## 9. Sales volume over time (sanity check on the 6-month window)

In [ ]:
# Adjust to your actual close-date column name, e.g. CloseDate
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
monthly_counts = df["CloseDate"].dt.to_period("M").value_counts().sort_index()

monthly_counts.plot(kind="bar", figsize=(10, 4), title="Sales per month")
plt.ylabel("count")
plt.show()


## 10. Notes / findings (fill in as you go)

- Target skew: ...
- Missingness worth addressing in Week 3: ...
- Outliers/data-quality issues spotted: ...
- Features that look most predictive so far: ...
- Anything odd about the monthly volume (e.g. a partial month at the edges of your
  download window that you may need to trim before the train/test split): ...
